In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

# importerer hele datasettet
weather_df_full = pd.read_csv("../raw_data/weather.csv")

# importerer stations og gjør samme prosedyre som i 01_explore_stations for at treningsdata
# i weather.csv skal være samme periode som stations.csv tross at data går over ulike perioder
stations_df_full = pd.read_csv("../raw_data/stations.csv")
stations_df_full["timestamp"] = pd.to_datetime(stations_df_full["timestamp"], utc=True).dt.tz_convert("Europe/Oslo")
stations_df_full = stations_df_full.sort_values(by="timestamp", ascending=True)

split_index = int(0.7*len(stations_df_full))
train_start = stations_df_full["timestamp"].min() 
train_end = stations_df_full.iloc[split_index]["timestamp"]

# korrigerer til riktig tidssone før videre analyse
# PS - vil endre dtype på timestamp til Europe/Oslo
weather_df_full["timestamp"] = pd.to_datetime(weather_df_full["timestamp"], utc=True).dt.tz_convert("Europe/Oslo")

# bruker kun treningsdata-perioden fra stations.csv i analyse av weather.csv
weather_df = weather_df_full[(weather_df_full["timestamp"] >= train_start) & (weather_df_full["timestamp"] <= train_end)]

In [2]:
# første 5 radene (i treningsdata)
head = weather_df.head()
head

,timestamp,temperature,precipitation,wind_speed
2424,2024-04-11 00:00:00+02:00,5.1,0.2,18.4
2425,2024-04-11 01:00:00+02:00,5.0,0.3,19.0
2426,2024-04-11 02:00:00+02:00,5.2,0.3,19.1
2427,2024-04-11 03:00:00+02:00,5.3,0.8,21.8
2428,2024-04-11 04:00:00+02:00,5.0,0.9,21.5


In [3]:
# siste 5 radene (i treningsdata)
tail = weather_df.tail()
tail

,timestamp,temperature,precipitation,wind_speed
9926,2025-02-17 14:00:00+01:00,-0.5,0.0,2.7
9927,2025-02-17 15:00:00+01:00,0.4,0.0,2.4
9928,2025-02-17 16:00:00+01:00,-0.1,0.0,1.4
9929,2025-02-17 17:00:00+01:00,-1.1,0.0,0.5
9930,2025-02-17 18:00:00+01:00,-2.5,0.0,0.3


In [4]:
# data info
info = weather_df.info()
info

<class 'pandas.core.frame.DataFrame'>
Index: 7507 entries, 2424 to 9930
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype                      
---  ------         --------------  -----                      
 0   timestamp      7507 non-null   datetime64[ns, Europe/Oslo]
 1   temperature    7430 non-null   float64                    
 2   precipitation  7441 non-null   float64                    
 3   wind_speed     7438 non-null   float64                    
dtypes: datetime64[ns, Europe/Oslo](1), float64(3)
memory usage: 293.2 KB


In [5]:
# data beskrivelse
description = weather_df.describe(include='all')
description

,timestamp,temperature,precipitation,wind_speed
count,7507,7430.000000,7441.000000,7438.000000
mean,2024-09-14 09:21:48.698548224+02:00,8.900094,0.319540,10.982280
min,2024-04-11 00:00:00+02:00,-12.600000,0.000000,0.000000
25%,2024-06-28 04:30:00+02:00,4.500000,0.000000,5.500000
50%,2024-09-14 09:00:00+02:00,9.400000,0.000000,9.400000
75%,2024-12-01 13:30:00+01:00,13.400000,0.200000,15.100000
max,2025-02-17 18:00:00+01:00,27.900000,9.600000,40.600000
std,NaN,6.634662,0.759092,6.983211


In [6]:
# hvor lenge målingene har holdt på (i treningsdata)
min = weather_df["timestamp"].min()
max = weather_df["timestamp"].max()

print("Målinger startet", min, "og sluttet", max)

Målinger startet 2024-04-11 00:00:00+02:00 og sluttet 2025-02-17 18:00:00+01:00


In [7]:
# sjekk for manglende verdier (i treningsdata)
isna = weather_df.isna().sum()
isna

timestamp         0
temperature      77
precipitation    66
wind_speed       69
dtype: int64

In [8]:
# sjekk for unike rader (i treningsdata)
nunique = weather_df.nunique()
nunique

timestamp        7507
temperature       391
precipitation      66
wind_speed        351
dtype: int64

In [9]:
# sjekker etter outliers / unormale verdier (i treningsdata)
outliers = weather_df.query("temperature < -40 or temperature > 40 or wind_speed < 0 or wind_speed > 45")
if outliers.empty:
    print("Ingen outliers / unormale verdier i treningsdata")

Ingen outliers / unormale verdier i treningsdata


In [10]:
# OpenAI's ChatGPT ble brukt til å formulere og/eller komme opp med ideer for deler av kode.
# Dette gjelder formateringen av temperatur til lett leselig format (samme blir brukt for vindfart).
# All kode som er hjulpet av AI er forstått i seg selv og ihht til resten av programmet.
# OpenAI - https://chatgpt.com - henta 10.09.25 

# finner laveste og høyeste målte temperaturer (i treningsdata)
lowest_temp = weather_df["temperature"].min()
highest_temp = weather_df["temperature"].max()

# finner tidsstempel for disse temperaturene
lowest_temp_time = weather_df[weather_df["temperature"] == lowest_temp]["timestamp"].dt.strftime("%d.%m.%y %H:%M").iloc[0]
highest_temp_time = weather_df[weather_df["temperature"] == highest_temp]["timestamp"].dt.strftime("%d.%m.%y %H:%M").iloc[0]

print("Laveste temperatur var", lowest_temp, "C, og inntraff", lowest_temp_time,
       "\nHøyeste temperatur var", highest_temp, "C, og inntraff", highest_temp_time)

Laveste temperatur var -12.6 C, og inntraff 05.01.25 04:00 
Høyeste temperatur var 27.9 C, og inntraff 05.09.24 16:00


In [11]:
# finner laveste og høyeste målte vindfart (i treningsdata)
lowest_wind = weather_df["wind_speed"].min()
highest_wind = weather_df["wind_speed"].max()
lowest_wind_time = weather_df[weather_df["wind_speed"] == lowest_wind]["timestamp"].dt.strftime("%d.%m.%y %H:%M").iloc[0]
highest_wind_time = weather_df[weather_df["wind_speed"] == highest_wind]["timestamp"].dt.strftime("%d.%m.%y %H:%M").iloc[0]

print("Laveste vindfart var", lowest_wind, "m/s, og inntraff først i målingene", lowest_wind_time,
       "\nHøyeste vindfart var", highest_wind, "m/s, og inntraff", highest_wind_time)

Laveste vindfart var 0.0 m/s, og inntraff først i målingene 27.04.24 10:00 
Høyeste vindfart var 40.6 m/s, og inntraff 21.10.24 08:00


In [12]:
# visualiserer temperatur over tid via lineplot
fig = px.line(weather_df, "timestamp", "temperature")
fig.update_layout(title="temperatur over tid", yaxis_title="temperatur (C)")
fig.show()

c:\Users\Jon\OneDrive - University of Bergen\Emne\3. semester\INF161\envs\INF161\Lib\site-packages\kaleido\__init__.py:14: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [13]:
# visualiserer vindfart over tid via lineplot
fig = px.line(weather_df, "timestamp", "wind_speed")
fig.update_layout(title="vindfart over tid", yaxis_title="vindfart (m/s)")
fig.show()

In [14]:
# visualiserer mengde nedbør over tid via lineplot
fig = px.line(weather_df, "timestamp", "precipitation")
fig.update_layout(title="nedbør over tid", yaxis_title="nedbør (mm)")

In [15]:
# visualiserer om været varierer særlig mellom dager i uka via boxplot
weather_df["day_of_week"] = weather_df["timestamp"].dt.day_name()

fig = px.box(weather_df, "day_of_week", "temperature")
fig.update_layout(title="temperaturforskjeller mellom dager i uka", xaxis_title="dag i uka",
                  yaxis_title="temperatur (C)")
fig.show()

C:\Users\Jon\AppData\Local\Temp\ipykernel_26864\2976637625.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [16]:
# visualiserer om været varierer særlig mellom dager i uka via boxplot
weather_df["month"] = weather_df["timestamp"].dt.month_name()

fig = px.box(weather_df, "month", "temperature")
fig.update_layout(title="temperaturforskjeller mellom måneder i året", xaxis_title="måned",
                  yaxis_title="temperatur")
fig.show()

C:\Users\Jon\AppData\Local\Temp\ipykernel_26864\2563683181.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [17]:
# visualiserer temperatur og vindfart via scatter
fig = px.scatter(weather_df, "temperature", "wind_speed")
fig.update_layout(title="temperatur vs vindfart", xaxis_title="temperatur (C)",
                  yaxis_title="vindfart (m/s)")
fig.show()